# 10 — Naive Predictors (drevalpy baselines)

All 6 naive baselines from drevalpy, adapted to the project's `pairs` / `folds` (one shared train, four test sets per fold) and evaluated with `evaluateMT` for consistency with the DL notebooks.

## Environment setup (Colab or local)

In [2]:
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/MyDrive/multiomics_project')
else:
    BASE_PATH = Path('..')

print(f"Running on {'Colab' if IN_COLAB else 'local'} | BASE_PATH = {BASE_PATH}")


Running on local | BASE_PATH = ..


## Imports and config

In [3]:
import json

import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr, linregress
from sklearn.metrics import r2_score, roc_auc_score


In [4]:
PROCESSED_DIR = BASE_PATH / "data" / "processed"
SPLITS_DIR = BASE_PATH / "data" / "splits"
RESULTS_DIR = BASE_PATH / "results" / "naive_pred"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

COL_CELL_LINE = "cell_line_name"
COL_DRUG = "drug_name"
COL_IC50 = "LN_IC50"
COL_CELLOSAURUS = "cellosaurus_id"
COL_TISSUE = "tissue"

SPLIT_TYPES = ["lpo", "lco", "ldo", "lto"]


## Load response pairs + splits (from notebook 04)

In [5]:
pairs = pd.read_parquet(PROCESSED_DIR / "response_pairs.parquet")

with open(SPLITS_DIR / "splits.json") as f:
    folds = json.load(f)

with open(SPLITS_DIR / "holdout_groups.json") as f:
    holdout_groups = json.load(f)

print(f"{len(pairs)} pairs loaded")
for fd in folds:
    print(f"fold {fd['fold']}: train={len(fd['train']):,} | "
          f"lco_test={len(fd['lco_test']):,} | ldo_test={len(fd['ldo_test']):,} | "
          f"lto_test={len(fd['lto_test']):,} | lpo_test={len(fd['lpo_test']):,}")


176197 pairs loaded
fold 0: train=107,421 | lco_test=17,470 | ldo_test=18,404 | lto_test=25,964 | lpo_test=20,613
fold 1: train=118,444 | lco_test=17,331 | ldo_test=18,515 | lto_test=13,579 | lpo_test=16,475
fold 2: train=97,262 | lco_test=18,173 | ldo_test=18,126 | lto_test=35,198 | lpo_test=23,832
fold 3: train=111,375 | lco_test=17,849 | ldo_test=18,140 | lto_test=21,451 | lpo_test=19,147
fold 4: train=103,126 | lco_test=17,762 | ldo_test=18,533 | lto_test=28,869 | lpo_test=21,721


## Evaluation metric

Same `evaluateMT` used across the DL notebooks, so naive baselines are directly comparable to model results.

In [6]:
import warnings

def evaluateMT(target, pred, threshold=None):
    """
    threshold: cutoff for binarizing target into sensitive/resistant for ROC-AUC.
    Defaults to median of target if not provided.
    """
    variance_real = np.var(target)
    variance_pred = np.var(pred)
    mses = ((target - pred) ** 2).mean(axis=0)
    rmse = np.sqrt(mses)

    pred_is_constant = np.isclose(variance_pred, 0)

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        if pred_is_constant:
            correlation, corr_p_value = np.nan, np.nan
            spearman_corr, spearman_p = np.nan, np.nan
            slope, intercept, r_value, lr_p_value, std_err = (np.nan,) * 5
        else:
            correlation, corr_p_value = pearsonr(target, pred)
            spearman_corr, spearman_p = spearmanr(target, pred)
            slope, intercept, r_value, lr_p_value, std_err = linregress(pred, target)

    r2 = r2_score(target, pred)

    if threshold is None:
        threshold = np.median(target)
    target_binary = (target < threshold).astype(int)
    if len(np.unique(target_binary)) == 2 and not pred_is_constant:
        roc_auc = roc_auc_score(target_binary, -pred)
    else:
        roc_auc = np.nan

    results = {
        'Correlation': round(correlation, 2) if not np.isnan(correlation) else correlation,
        'Corr p-value': round(corr_p_value, 4) if not np.isnan(corr_p_value) else corr_p_value,
        'Spearman': round(spearman_corr, 2) if not np.isnan(spearman_corr) else spearman_corr,
        'MSE': round(mses, 2),
        'RMSE': round(rmse, 2),
        'R2': round(r2, 2),
        'ROC-AUC': round(roc_auc, 2) if not np.isnan(roc_auc) else roc_auc,
        'Slope': round(slope, 2) if not np.isnan(slope) else slope,
        'Standard error': round(std_err, 2) if not np.isnan(std_err) else std_err,
        'Variance Real': round(variance_real, 2),
        'Variance Pred': round(variance_pred, 2)
    }
    return pd.DataFrame([results])

## Naive predictor strategies

All 6 naive baselines from drevalpy's `naive_pred.py`:

| Model | Strategy |
|---|---|
| `NaivePredictor` | overall dataset mean |
| `NaiveDrugMeanPredictor` | mean per drug, fallback to dataset mean |
| `NaiveCellLineMeanPredictor` | mean per cell line, fallback to dataset mean |
| `NaiveTissueMeanPredictor` | mean per tissue, fallback to dataset mean |
| `NaiveTissueDrugMeanPredictor` | mean per (tissue, drug), fallback to dataset mean |
| `NaiveMeanEffectsPredictor` | dataset_mean + cell_line_effect + drug_effect (ANOVA-style) — drevalpy flags this as the strongest naive baseline |

All stats are computed from `train` only (shared across all 4 split types per fold).

In [7]:
def naive_means(train: pd.DataFrame) -> dict:
    """Computes all naive lookup tables from the training set."""
    dataset_mean = train[COL_IC50].mean()
    drug_means = train.groupby(COL_DRUG)[COL_IC50].mean().to_dict()
    cell_means = train.groupby(COL_CELLOSAURUS)[COL_IC50].mean().to_dict()
    tissue_means = train.groupby(COL_TISSUE)[COL_IC50].mean().to_dict()
    tissue_drug_means = train.groupby([COL_TISSUE, COL_DRUG])[COL_IC50].mean().to_dict()
    return {
        "dataset_mean": dataset_mean,
        "drug_means": drug_means,
        "cell_means": cell_means,
        "tissue_means": tissue_means,
        "tissue_drug_means": tissue_drug_means,
        "cell_line_effects": {cl: m - dataset_mean for cl, m in cell_means.items()},
        "drug_effects": {d: m - dataset_mean for d, m in drug_means.items()},
    }


def predict_naive(stats: dict, test: pd.DataFrame, strategy: str) -> np.ndarray:
    if strategy == "overall":
        return np.full(len(test), stats["dataset_mean"])
    if strategy == "drug":
        return test[COL_DRUG].map(stats["drug_means"]).fillna(stats["dataset_mean"]).to_numpy()
    if strategy == "cell_line":
        return test[COL_CELLOSAURUS].map(stats["cell_means"]).fillna(stats["dataset_mean"]).to_numpy()
    if strategy == "tissue":
        return test[COL_TISSUE].map(stats["tissue_means"]).fillna(stats["dataset_mean"]).to_numpy()
    if strategy == "tissue_drug":
        keys = list(zip(test[COL_TISSUE], test[COL_DRUG]))
        return np.array([stats["tissue_drug_means"].get(k, stats["dataset_mean"]) for k in keys])
    if strategy == "mean_effects":
        cl_eff = test[COL_CELLOSAURUS].map(stats["cell_line_effects"]).fillna(0.0)
        drug_eff = test[COL_DRUG].map(stats["drug_effects"]).fillna(0.0)
        return (stats["dataset_mean"] + cl_eff + drug_eff).to_numpy()
    raise ValueError(f"Unknown strategy: {strategy}")


NAIVE_STRATEGIES = {
    "NaivePredictor": "overall",
    "NaiveDrugMeanPredictor": "drug",
    "NaiveCellLineMeanPredictor": "cell_line",
    "NaiveTissueMeanPredictor": "tissue",
    "NaiveTissueDrugMeanPredictor": "tissue_drug",
    "NaiveMeanEffectsPredictor": "mean_effects",
}


## Run all naive predictors across all folds and split types

In [8]:
naive_results = []
for fold_i, fold in enumerate(folds):
    train = pairs.loc[np.array(fold["train"])]
    stats = naive_means(train)

    for split_type in SPLIT_TYPES:
        test = pairs.loc[np.array(fold[f"{split_type}_test"])]
        y_true = test[COL_IC50].to_numpy()

        for model_name, strategy in NAIVE_STRATEGIES.items():
            y_pred = predict_naive(stats, test, strategy)
            df = evaluateMT(y_true, y_pred)
            df.insert(0, "Fold", fold_i)
            df.insert(1, "Split", split_type.upper())
            df.insert(2, "Model", model_name)
            naive_results.append(df)

naive_results_df = pd.concat(naive_results, ignore_index=True)
naive_results_df


,Fold,Split,Model,Correlation,Corr p-value,Spearman,MSE,RMSE,R2,ROC-AUC,Slope,Standard error,Variance Real,Variance Pred
0,0,LPO,NaivePredictor,NaN,NaN,NaN,7.29,2.70,-0.00,NaN,NaN,NaN,7.27,0.00
1,0,LPO,NaiveDrugMeanPredictor,0.79,0.0,0.73,2.72,1.65,0.63,0.86,1.00,0.01,7.27,4.53
2,0,LPO,NaiveCellLineMeanPredictor,0.27,0.0,0.28,6.74,2.60,0.07,0.63,0.99,0.02,7.27,0.56
3,0,LPO,NaiveTissueMeanPredictor,0.15,0.0,0.17,7.12,2.67,0.02,0.58,0.99,0.04,7.27,0.17
4,0,LPO,NaiveTissueDrugMeanPredictor,0.74,0.0,0.70,3.28,1.81,0.55,0.85,0.98,0.01,7.27,4.14
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,4,LTO,NaiveDrugMeanPredictor,0.79,0.0,0.72,3.58,1.89,0.52,0.85,1.02,0.00,7.53,4.44
116,4,LTO,NaiveCellLineMeanPredictor,NaN,NaN,NaN,8.20,2.86,-0.09,NaN,NaN,NaN,7.53,0.00
117,4,LTO,NaiveTissueMeanPredictor,NaN,NaN,NaN,8.20,2.86,-0.09,NaN,NaN,NaN,7.53,0.00
118,4,LTO,NaiveTissueDrugMeanPredictor,NaN,NaN,NaN,8.20,2.86,-0.09,NaN,NaN,NaN,7.53,0.00


In [9]:
naive_results_df[naive_results_df["Model"] == "NaiveTissueDrugMeanPredictor"]

,Fold,Split,Model,Correlation,Corr p-value,Spearman,MSE,RMSE,R2,ROC-AUC,Slope,Standard error,Variance Real,Variance Pred
4,0,LPO,NaiveTissueDrugMeanPredictor,0.74,0.0,0.70,3.28,1.81,0.55,0.85,0.98,0.01,7.27,4.14
10,0,LCO,NaiveTissueDrugMeanPredictor,0.74,0.0,0.70,3.23,1.80,0.55,0.85,0.98,0.01,7.16,4.11
16,0,LDO,NaiveTissueDrugMeanPredictor,NaN,NaN,NaN,6.52,2.55,-0.00,NaN,NaN,NaN,6.52,0.00
22,0,LTO,NaiveTissueDrugMeanPredictor,NaN,NaN,NaN,8.42,2.90,-0.12,NaN,NaN,NaN,7.54,0.00
28,1,LPO,NaiveTissueDrugMeanPredictor,0.79,0.0,0.73,2.76,1.66,0.62,0.86,1.00,0.01,7.33,4.61
34,1,LCO,NaiveTissueDrugMeanPredictor,0.79,0.0,0.72,2.77,1.67,0.62,0.86,0.97,0.01,7.23,4.74
40,1,LDO,NaiveTissueDrugMeanPredictor,NaN,NaN,NaN,7.35,2.71,-0.00,NaN,NaN,NaN,7.34,0.00
46,1,LTO,NaiveTissueDrugMeanPredictor,NaN,NaN,NaN,6.92,2.63,-0.01,NaN,NaN,NaN,6.84,0.00
52,2,LPO,NaiveTissueDrugMeanPredictor,0.74,0.0,0.68,3.31,1.82,0.54,0.83,0.98,0.01,7.20,4.08
58,2,LCO,NaiveTissueDrugMeanPredictor,0.76,0.0,0.70,3.11,1.76,0.58,0.84,0.97,0.01,7.34,4.49


## Summary: mean across folds, per model x split

The single most useful view for comparing against DL results — averages out fold noise.

In [10]:
summary_df = (
    naive_results_df
    .groupby(["Model", "Split"])[["Correlation", "Spearman", "RMSE", "R2", "ROC-AUC"]]
    .mean()
    .round(3)
    .reset_index()
)
summary_df


,Model,Split,Correlation,Spearman,RMSE,R2,ROC-AUC
0,NaiveCellLineMeanPredictor,LCO,NaN,NaN,2.692,-0.006,NaN
1,NaiveCellLineMeanPredictor,LDO,0.284,0.296,2.470,0.048,0.636
2,NaiveCellLineMeanPredictor,LPO,0.276,0.286,2.586,0.072,0.632
3,NaiveCellLineMeanPredictor,LTO,NaN,NaN,2.774,-0.056,NaN
4,NaiveDrugMeanPredictor,LCO,0.790,0.726,1.656,0.620,0.856
5,NaiveDrugMeanPredictor,LDO,NaN,NaN,2.568,-0.032,NaN
6,NaiveDrugMeanPredictor,LPO,0.788,0.724,1.660,0.620,0.858
7,NaiveDrugMeanPredictor,LTO,0.794,0.738,1.766,0.574,0.862
8,NaiveMeanEffectsPredictor,LCO,0.790,0.726,1.656,0.620,0.856
9,NaiveMeanEffectsPredictor,LDO,0.284,0.296,2.470,0.048,0.636


In [11]:
summary_df[summary_df["Model"] == "NaiveTissueDrugMeanPredictor"]

,Model,Split,Correlation,Spearman,RMSE,R2,ROC-AUC
16,NaiveTissueDrugMeanPredictor,LCO,0.762,0.708,1.744,0.578,0.852
17,NaiveTissueDrugMeanPredictor,LDO,NaN,NaN,2.568,-0.032,NaN
18,NaiveTissueDrugMeanPredictor,LPO,0.752,0.700,1.776,0.562,0.846
19,NaiveTissueDrugMeanPredictor,LTO,NaN,NaN,2.774,-0.056,NaN


## Save results

In [12]:
naive_results_df.to_parquet(RESULTS_DIR / "naive_predictors_per_fold.parquet")
summary_df.to_csv(RESULTS_DIR / "naive_predictors_summary.csv", index=False)
print(f"Saved to {RESULTS_DIR}")


Saved to ../results/naive_pred
